# Graph Neural Networks — Implementations

Both layers again on tensors. These are forward-only layer algorithms, so the equivalence fixture is a single deterministic forward pass on a literal 7-node graph, with features and Xavier draws coming from named NumPy generators in every lane. No library lanes in this topic: the sandbox has no torch_geometric and sklearn has no GNN estimators, so hand-written torch — exactly what these lanes are — is already the practitioner version.

## 20_gcn_layer

Normalized neighborhood averaging, then a shared linear map. No library lane: torch_geometric is not in the sandbox and sklearn has no GCN, so there is no honest library comparison — the torch lane is the practitioner implementation here.

### torch

The setup's `get_laplacian` and the `gcn_normalize` step become three tensor lines each, and the layer keeps the scratch `(A_norm @ H) @ W` order so the numbers match. **What torch adds:** `requires_grad` on `W` — one `backward()` yields the gradient the scratch layer never wrote, and a check confirms it equals the hand chain rule `(SH)ᵀ 1[pre>0]`.

In [ ]:
import numpy as np
import torch

# hints:
# 1. Degree vector -> pow(-0.5) -> torch.diag: gcn_normalize is three tensor lines.
# 2. Keep the (A_norm @ H) @ W evaluation order the NumPy expression already has.
# 3. Draw W with NumPy's rng, then torch.tensor(..., requires_grad=True), float64.
# 4. sum().backward() gives dW for free; the hand formula is (SH)^T 1[pre>0].


def get_laplacian(A):
    """L = D - A on tensors; still symmetric, still positive semi-definite."""
    D = torch.diag(A.sum(dim=1))
    return D - A


def gcn_normalize(A):
    """D^-1/2 (A + I) D^-1/2 — the renormalization trick, as tensor ops."""
    A_tilde = A + torch.eye(A.shape[0], dtype=A.dtype)
    d = A_tilde.sum(dim=1)
    D_inv_sqrt = torch.diag(d.pow(-0.5))
    return D_inv_sqrt @ A_tilde @ D_inv_sqrt


def relu(x):
    return torch.clamp(x, min=0)


def softmax(x, axis=-1):
    return torch.softmax(x, dim=axis)


class GCNLayer:
    """The scratch layer on tensors: same Xavier draw (from NumPy's generator,
    so the numbers match), same three matmuls, plus requires_grad on W."""

    def __init__(self, in_features, out_features, rng):
        limit = np.sqrt(6 / (in_features + out_features))
        W = rng.uniform(-limit, limit, size=(in_features, out_features))
        self.W = torch.tensor(W, dtype=torch.float64, requires_grad=True)

    def forward(self, H, A_norm):
        return A_norm @ H @ self.W


In [ ]:
# exports: H_out, grad_W, A_norm_fix
_edges_eq = np.array([[0, 1], [0, 2], [1, 2], [2, 3], [3, 4], [4, 5], [4, 6], [5, 6]])
_A_np_eq = np.zeros((7, 7))
_A_np_eq[_edges_eq[:, 0], _edges_eq[:, 1]] = 1.0
_A_np_eq[_edges_eq[:, 1], _edges_eq[:, 0]] = 1.0
_A_eq = torch.as_tensor(_A_np_eq)
_H_eq = torch.as_tensor(np.random.default_rng(7).normal(size=(7, 5)))
_layer_eq = GCNLayer(5, 3, np.random.default_rng(21))
_A_norm_eq = gcn_normalize(_A_eq)
_out_eq = relu(_layer_eq.forward(_H_eq, _A_norm_eq))
_out_eq.sum().backward()
H_out = _out_eq.detach().numpy()
grad_W = _layer_eq.W.grad.numpy()
A_norm_fix = _A_norm_eq.numpy()
print("H_out shape:", H_out.shape, "| grad_W shape:", grad_W.shape)


In [ ]:
# The normalized operator is symmetric with spectrum in [-1, 1] — a low-pass filter.
assert torch.allclose(_A_norm_eq, _A_norm_eq.T, atol=1e-12), "D^-1/2 (A+I) D^-1/2 is symmetric"
_eigs_eq = torch.linalg.eigvalsh(_A_norm_eq)
assert float(_eigs_eq.max()) <= 1.0 + 1e-10, "largest eigenvalue of A_norm is 1"
assert float(_eigs_eq.min()) >= -1.0 - 1e-10, "spectrum of A_norm stays above -1"

# The Laplacian mirrored from setup keeps its defining property.
_L_eq = get_laplacian(_A_eq)
assert float(torch.linalg.eigvalsh(_L_eq).min()) > -1e-10, "L = D - A is positive semi-definite"

# Autograd reproduces the hand chain rule for sum(relu(S H W)).
_pre_eq = _A_norm_eq @ _H_eq @ _layer_eq.W.detach()
_hand_eq = (_A_norm_eq @ _H_eq).T @ (_pre_eq > 0).to(torch.float64)
assert torch.allclose(_layer_eq.W.grad, _hand_eq, atol=1e-12), "autograd matches (SH)^T 1[pre>0]"

# Permutation equivariance: relabel the nodes and the outputs relabel with them.
_perm_eq = torch.tensor([3, 1, 4, 0, 2, 6, 5])
_A_p_eq = _A_eq[_perm_eq][:, _perm_eq]
_out_p_eq = relu(_layer_eq.forward(_H_eq[_perm_eq], gcn_normalize(_A_p_eq)))
assert torch.allclose(_out_p_eq.detach(), _out_eq.detach()[_perm_eq], atol=1e-12), \
    "GCN output is permutation-equivariant"


## 20_gat_layer

Attention restricted to the graph: softmax over each node's neighborhood. No library lane for the same reason as the GCN — no torch_geometric in the sandbox, and nothing in sklearn to compare against honestly.

### torch

The same all-pairs construction (`repeat_interleave`/`repeat` for `np.repeat`/`np.tile`), the same LeakyReLU slope, and the same additive `-1e9` mask rather than `-inf`, so non-edges underflow to exactly zero attention in both lanes. **What torch adds:** autograd through the masked softmax — the checks verify `a.grad` against central differences, a derivative nobody wants to write by hand.

In [ ]:
import numpy as np
import torch

# hints:
# 1. repeat_interleave(N, dim=0) is np.repeat; .repeat(N, 1) is np.tile — same pairs.
# 2. torch.where(e > 0, e, alpha * e) is the LeakyReLU with the notebook's exact slope.
# 3. Add the same -1e9 mask the scratch adds (not -inf); exp underflows to exact zeros.
# 4. torch.softmax subtracts the row max like the scratch softmax — deltas stay ~1e-16.
# 5. Draw W and a from one NumPy generator in the scratch order: W first, then a.


class GATLayer:
    """Single-head graph attention on tensors — the scratch layer line by line,
    with W and a drawn from NumPy so every lane attends with the same numbers."""

    def __init__(self, in_features, out_features, rng, alpha=0.2):
        limit = np.sqrt(6 / (in_features + out_features))
        self.W = torch.tensor(rng.uniform(-limit, limit, size=(in_features, out_features)),
                              dtype=torch.float64, requires_grad=True)
        self.a = torch.tensor(rng.uniform(-limit, limit, size=(2 * out_features, 1)),
                              dtype=torch.float64, requires_grad=True)
        self.alpha = alpha  # LeakyReLU slope

    def forward(self, H, A):
        N = H.shape[0]
        WH = H @ self.W  # (N, out_features)

        # All (i, j) concatenations [Wh_i || Wh_j], exactly as the NumPy lane builds them
        WH_i = WH.repeat_interleave(N, dim=0).reshape(N, N, -1)
        WH_j = WH.repeat(N, 1).reshape(N, N, -1)
        WH_concat = torch.cat([WH_i, WH_j], dim=-1)  # (N, N, 2*out_features)

        e = (WH_concat @ self.a).squeeze(-1)  # (N, N)
        e = torch.where(e > 0, e, self.alpha * e)  # LeakyReLU

        # Mask non-neighbors with the scratch lane's additive -1e9 (self-loops kept)
        A_tilde = A + torch.eye(N, dtype=A.dtype)
        mask = torch.where(A_tilde > 0, torch.zeros_like(A_tilde),
                           torch.full_like(A_tilde, -1e9))
        e = e + mask

        attention = torch.softmax(e, dim=1)  # (N, N)
        H_out = attention @ WH
        return H_out, attention


In [ ]:
# exports: H_out, attention
_edges_eq = np.array([[0, 1], [0, 2], [1, 2], [2, 3], [3, 4], [4, 5], [4, 6], [5, 6]])
_A_np_eq = np.zeros((7, 7))
_A_np_eq[_edges_eq[:, 0], _edges_eq[:, 1]] = 1.0
_A_np_eq[_edges_eq[:, 1], _edges_eq[:, 0]] = 1.0
_A_eq = torch.as_tensor(_A_np_eq)
_H_eq = torch.as_tensor(np.random.default_rng(7).normal(size=(7, 5)))
_gat_eq = GATLayer(5, 3, np.random.default_rng(33))
_out_eq, _att_eq = _gat_eq.forward(_H_eq, _A_eq)
H_out = _out_eq.detach().numpy()
attention = _att_eq.detach().numpy()
print("attention row sums:", np.round(attention.sum(axis=1), 6))


In [ ]:
# Attention is a distribution over each node's neighborhood.
assert torch.allclose(_att_eq.sum(dim=1), torch.ones(7, dtype=torch.float64), atol=1e-12), \
    "each attention row sums to 1"

# The mask really works: nothing leaks to non-neighbors, self-loops stay in.
_A_tilde_np_eq = _A_np_eq + np.eye(7)
assert float(np.max(attention[_A_tilde_np_eq == 0])) < 1e-12, "zero attention on non-edges"
assert float(np.min(attention[_A_tilde_np_eq > 0])) > 0.0, "neighbors and self get positive mass"

# Permutation equivariance: relabelled nodes attend to the same relabelled neighbors.
_perm_eq = torch.tensor([3, 1, 4, 0, 2, 6, 5])
_out_p_eq, _att_p_eq = _gat_eq.forward(_H_eq[_perm_eq], _A_eq[_perm_eq][:, _perm_eq])
assert torch.allclose(_att_p_eq.detach(), _att_eq.detach()[_perm_eq][:, _perm_eq], atol=1e-10), \
    "attention matrix is permutation-equivariant"

# Autograd through the masked softmax agrees with central differences on a.
_out_eq.sum().backward()
_g_a_eq = _gat_eq.a.grad.clone()
_a0_eq = _gat_eq.a.detach().clone()
_eps_eq = 1e-6
_fd_errs_eq = []
for _k_eq in range(3):
    with torch.no_grad():
        _gat_eq.a[_k_eq, 0] = _a0_eq[_k_eq, 0] + _eps_eq
    _plus_eq = float(_gat_eq.forward(_H_eq, _A_eq)[0].sum())
    with torch.no_grad():
        _gat_eq.a[_k_eq, 0] = _a0_eq[_k_eq, 0] - _eps_eq
    _minus_eq = float(_gat_eq.forward(_H_eq, _A_eq)[0].sum())
    with torch.no_grad():
        _gat_eq.a[_k_eq, 0] = _a0_eq[_k_eq, 0]
    _fd_errs_eq.append(abs((_plus_eq - _minus_eq) / (2 * _eps_eq) - float(_g_a_eq[_k_eq, 0])))
assert max(_fd_errs_eq) < 1e-5, "autograd gradient matches finite differences"
